In [81]:
import sympy as sp
import numpy as np

In [82]:
class NonlinearSolver:
    def __init__(self, equations, variables, x0, alpha=0.5):
        self.vars = sp.Matrix(variables)
        self.n = len(variables)
        self.x0 = np.array(x0, float)
        self.alpha = alpha
        
        self.F = sp.Matrix(equations)
        self.J = self.F.jacobian(self.vars)
        self.Phi = sum(f**2 for f in self.F)
        self.Grad = sp.Matrix([self.Phi.diff(v) for v in self.vars])
        self.IterForm = self.vars - self.alpha * self.F
        
        self.f_num = sp.lambdify(self.vars, self.F, 'numpy')
        self.j_num = sp.lambdify(self.vars, self.J, 'numpy')
        self.g_num = sp.lambdify(self.vars, self.Grad, 'numpy')
        self.iter_num = sp.lambdify(self.vars, self.IterForm, 'numpy')

    def _solve(self, name, step_func, eps=0.001, max_iter=50):
        x = self.x0.copy()
        print(f"{'Iter':<5} | " + " | ".join([f"x{i+1}".center(10) for i in range(self.n)]) + f" | {'Error':^10}")
        
        for k in range(max_iter):
            x_new = step_func(x)
            err = np.max(np.abs(x_new - x))
            vals = " | ".join([f"{x_new[i]:^10.5f}" for i in range(self.n)])
            print(f"{k+1:<5} | {vals} | {err:^10.5f}")
            
            if err < eps:
                print(f"\nV Решение: {[round(v, 5) for v in x_new]}")
                return x_new
            x = x_new
            
        print("\nX Не сошлось")
        return None

    def simple_iteration(self, eps=0.001):
        return self._solve("Простые итерации", 
                           lambda x: np.array(self.iter_num(*x)).flatten(), eps)

    def seidel(self, eps=0.001):
        def step(x):
            x_new = x.copy()
            for i in range(self.n):
                tmp = x_new.copy()
                res = np.array(self.iter_num(*tmp)).flatten()
                x_new[i] = res[i]
            return x_new
        return self._solve("Зейдель", step, eps)

    def newton(self, eps=0.001):
        def step(x):
            F = np.array(self.f_num(*x)).flatten()
            J = np.array(self.j_num(*x)).flatten().reshape(self.n, self.n)
            return x + np.linalg.solve(J, -F)
        return self._solve("Ньютон", step, eps)

    def steepest_descent(self, eps=0.001):
        def step(x):
            G = np.array(self.g_num(*x)).flatten()
            alpha = 0.1
            for _ in range(10):
                xn = x - alpha * G
                if np.sum(np.array(self.f_num(*xn)).flatten()**2) < np.sum(np.array(self.f_num(*x)).flatten()**2):
                    break
                alpha *= 0.5
            return x - alpha * G
        return self._solve("Наискорейший спуск", step, eps)

In [83]:
x, y = sp.symbols('x y')

equations = [sp.sin(x + 0.5) - y - 1, sp.cos(y - 2) + x]

solver = NonlinearSolver(equations, [x, y], [0.0, 0.0], alpha=0.5)

In [84]:
solver.simple_iteration()

Iter  |     x1     |     x2     |   Error   
1     |  0.26029   |  0.20807   |  0.26029  
2     |  0.51976   |  0.18760   |  0.25947  
3     |  0.68757   |  0.04735   |  0.16781  
4     |  0.74751   |  -0.11011  |  0.15746  
5     |  0.71835   |  -0.22709  |  0.11698  
6     |  0.63554   |  -0.28118  |  0.08281  
7     |  0.54157   |  -0.27289  |  0.09397  
8     |  0.47353   |  -0.22077  |  0.06804  
9     |  0.44971   |  -0.15495  |  0.06582  
10    |  0.46561   |  -0.10406  |  0.05089  
11    |  0.50238   |  -0.08269  |  0.03677  
12    |  0.53966   |  -0.08897  |  0.03728  
13    |  0.56406   |  -0.11115  |  0.02440  
14    |  0.57132   |  -0.13596  |  0.02481  
15    |  0.56442   |  -0.15384  |  0.01788  
16    |  0.55025   |  -0.16077  |  0.01418  
17    |  0.53609   |  -0.15772  |  0.01416  
18    |  0.52702   |  -0.14886  |  0.00907  
19    |  0.52471   |  -0.13917  |  0.00969  
20    |  0.52784   |  -0.13239  |  0.00678  
21    |  0.53355   |  -0.13004  |  0.00571  
22    |  0

array([ 0.53647306, -0.13761   ])

In [85]:
solver.seidel()

Iter  |     x1     |     x2     |   Error   
1     |  0.26029   |  0.07793   |  0.26029  
2     |  0.45469   |  0.02263   |  0.19440  
3     |  0.55794   |  -0.05860  |  0.10325  
4     |  0.59296   |  -0.12074  |  0.06214  
5     |  0.58860   |  -0.15372  |  0.03298  
6     |  0.56875   |  -0.16286  |  0.01985  
7     |  0.54902   |  -0.15833  |  0.01973  
8     |  0.53639   |  -0.14937  |  0.01263  
9     |  0.53142   |  -0.14166  |  0.00771  
10    |  0.53157   |  -0.13727  |  0.00439  
11    |  0.53388   |  -0.13588  |  0.00231  
12    |  0.53630   |  -0.13629  |  0.00241  
13    |  0.53789   |  -0.13732  |  0.00159  
14    |  0.53857   |  -0.13825  |  0.00093  

V Решение: [0.53857, -0.13825]


array([ 0.53856596, -0.1382506 ])

In [86]:
solver.newton()

Iter  |     x1     |     x2     |   Error   
1     |  0.49472   |  -0.08641  |  0.49472  
2     |  0.53786   |  -0.13789  |  0.05148  
3     |  0.53785   |  -0.13868  |  0.00079  

V Решение: [0.53785, -0.13868]


array([ 0.53785311, -0.13868456])

In [87]:
solver.steepest_descent()

Iter  |     x1     |     x2     |   Error   
1     |  0.17460   |  -0.02843  |  0.17460  
2     |  0.28224   |  -0.04988  |  0.10764  
3     |  0.35278   |  -0.06721  |  0.07054  
4     |  0.40112   |  -0.08143  |  0.04834  
5     |  0.43535   |  -0.09305  |  0.03423  
6     |  0.46017   |  -0.10248  |  0.02482  
7     |  0.47850   |  -0.11005  |  0.01833  
8     |  0.49223   |  -0.11610  |  0.01373  
9     |  0.50262   |  -0.12091  |  0.01039  
10    |  0.51055   |  -0.12471  |  0.00793  
11    |  0.51664   |  -0.12771  |  0.00609  
12    |  0.52134   |  -0.13008  |  0.00470  
13    |  0.52497   |  -0.13193  |  0.00364  
14    |  0.52779   |  -0.13339  |  0.00282  
15    |  0.52999   |  -0.13454  |  0.00220  
16    |  0.53170   |  -0.13543  |  0.00171  
17    |  0.53304   |  -0.13614  |  0.00134  
18    |  0.53409   |  -0.13669  |  0.00104  
19    |  0.53490   |  -0.13712  |  0.00082  

V Решение: [0.5349, -0.13712]


array([ 0.53490345, -0.13711923])